In [0]:
#  SCD TYPE 2 DIMENSIONS + CDC


from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# UNITY CATALOG CONFIGURATION
CATALOG = "retail_demo"
RAW_SCHEMA = "raw"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

# PROJECT PATH
BASE_PATH = (
    "/Volumes/retail_demo/raw/retail_files/"
    "retail_delta_project"
)
print(f"Catalog       : {CATALOG}")
print(f"Raw Schema    : {RAW_SCHEMA}")
print(f"Silver Schema : {SILVER_SCHEMA}")
print(f"Gold Schema   : {GOLD_SCHEMA}")
print(f"Base Path     : {BASE_PATH}")

Catalog       : retail_demo
Raw Schema    : raw
Silver Schema : silver
Gold Schema   : gold
Base Path     : /Volumes/retail_demo/raw/retail_files/retail_delta_project


In [0]:
#  VERIFY / ALIGN SOURCE TABLES

source_tables = [
    "customers_bronze",
    "products_bronze",
    "stores_bronze",
    "bronze_customers_incremental",
    "bronze_products_incremental",
    "silver1_customers_clean",
    "silver1_products_clean",
    "silver1_orders_clean",
    "silver1_stores_clean"
]

for table_name in source_tables:

    if "." in table_name:
        if table_name.startswith("silver"):
            full_name = f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
        else:
            full_name = f"{CATALOG}.{RAW_SCHEMA}.{table_name}"
    else:
        if table_name.startswith("silver"):
            full_name = f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
        else:
            full_name = f"{CATALOG}.{RAW_SCHEMA}.{table_name}"

    try:
        count = spark.table(full_name).count()
        print(f"✓ {full_name:<65} {count:,} rows")
    except Exception:
        print(f"✗ {full_name:<65} NOT FOUND")

✓ retail_demo.raw.customers_bronze                                  2,560 rows
✓ retail_demo.raw.products_bronze                                   830 rows
✓ retail_demo.raw.stores_bronze                                     80 rows
✓ retail_demo.raw.bronze_customers_incremental                      630 rows
✓ retail_demo.raw.bronze_products_incremental                       180 rows
✓ retail_demo.silver.silver1_customers_clean                        577 rows
✓ retail_demo.silver.silver1_products_clean                         166 rows
✓ retail_demo.silver.silver1_orders_clean                           5,716 rows
✓ retail_demo.silver.silver1_stores_clean                           75 rows


In [0]:
#  PREPARE HISTORICAL CUSTOMER BASE

customers_batch = spark.table(
    f"{CATALOG}.{RAW_SCHEMA}.customers_bronze"
)
print("Bronze batch rows:", customers_batch.count())

customers_batch.printSchema()

display(
    customers_batch.limit(10)
)

Bronze batch rows: 2560
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_type: string (nullable = true)



customer_id,customer_name,city,segment,gender,signup_date,status,_ingested_at,_source_file,_source_type
C00001,Customer_1,Delhi,Regular,Other,2024-10-08,active,2026-08-12T17:21:58.800Z,customers_batch.csv,batch
C00002,Customer_2,Bengaluru,Silver,Other,2024-04-14,active,2026-08-12T17:21:58.800Z,customers_batch.csv,batch
C00003,Customer_3,Gurugram,Platinum,M,2024-01-31,active,2026-08-12T17:21:58.800Z,customers_batch.csv,batch
C00004,Customer_4,Bengaluru,Silver,Other,2025-09-08,active,2026-08-12T17:21:58.800Z,customers_batch.csv,batch
C00005,Customer_5,Ahmedabad,Silver,Other,2025-10-27,inactive,2026-08-12T17:21:58.800Z,customers_batch.csv,batch
C00006,Customer_6,Bengaluru,Platinum,Other,2024-10-11,active,2026-08-12T17:21:58.800Z,customers_batch.csv,batch
C00007,Customer_7,Mumbai,Platinum,F,2024-10-11,active,2026-08-12T17:21:58.800Z,customers_batch.csv,batch
C00008,Customer_8,Bengaluru,Gold,M,2024-04-04,inactive,2026-08-12T17:21:58.800Z,customers_batch.csv,batch
C00009,Customer_9,Delhi,Gold,F,2025-09-10,active,2026-08-12T17:21:58.800Z,customers_batch.csv,batch
C00010,Customer_10,Jaipur,Platinum,Other,2024-05-07,inactive,2026-08-12T17:21:58.800Z,customers_batch.csv,batch


In [0]:
#  PREPARE INITIAL CUSTOMER DIMENSION

customer_base = (
    customers_batch
    .select(
        "customer_id",
        "customer_name",
        "city",
        "segment",
        "gender",
        "signup_date",
        "status"
    )
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("customer_name", F.trim(F.col("customer_name")))
    .withColumn("city", F.trim(F.col("city")))
    .withColumn("segment", F.trim(F.col("segment")))
    .withColumn("gender", F.trim(F.col("gender")))
    .withColumn("status", F.trim(F.col("status")))
    .withColumn(
        "signup_date",
        F.coalesce(
            F.expr(
                "try_to_date(trim(signup_date), 'yyyy-MM-dd')"
            ),
            F.expr(
                "try_to_date(trim(signup_date), 'dd-MM-yyyy')"
            )
        )
    )
    # Handle categorical nulls

    .withColumn(
        "city",
        F.when(
            F.col("city").isNull() |
            (F.trim(F.col("city")) == ""),
            F.lit("Unknown")
        ).otherwise(F.col("city"))
    )
    .withColumn(
        "segment",
        F.when(
            F.col("segment").isNull() |
            (F.trim(F.col("segment")) == ""),
            F.lit("Unknown")
        ).otherwise(F.col("segment"))
    )
    .withColumn(
        "gender",
        F.when(
            F.col("gender").isNull() |
            (F.trim(F.col("gender")) == ""),
            F.lit("Unknown")
        ).otherwise(F.col("gender"))
    )
    .withColumn(
        "status",
        F.when(
            F.col("status").isNull() |
            (F.trim(F.col("status")) == ""),
            F.lit("Unknown")
        ).otherwise(F.col("status"))
    )
    .filter(
        F.col("customer_id").isNotNull()
        & (F.col("customer_id") != "")
        & F.col("signup_date").isNotNull()
    )
    .dropDuplicates(["customer_id"])
)
print("Rows:", customer_base.count())

print(
    "Distinct customer IDs:",
    customer_base.select("customer_id").distinct().count()
)

display(customer_base.limit(10))

Rows: 2475
Distinct customer IDs: 2475


customer_id,customer_name,city,segment,gender,signup_date,status
C00001,Customer_1,Delhi,Regular,Other,2024-10-08,active
C00002,Customer_2,Bengaluru,Silver,Other,2024-04-14,active
C00003,Customer_3,Gurugram,Platinum,M,2024-01-31,active
C00004,Customer_4,Bengaluru,Silver,Other,2025-09-08,active
C00005,Customer_5,Ahmedabad,Silver,Other,2025-10-27,inactive
C00006,Customer_6,Bengaluru,Platinum,Other,2024-10-11,active
C00007,Customer_7,Mumbai,Platinum,F,2024-10-11,active
C00008,Customer_8,Bengaluru,Gold,M,2024-04-04,inactive
C00009,Customer_9,Delhi,Gold,F,2025-09-10,active
C00010,Customer_10,Jaipur,Platinum,Other,2024-05-07,inactive


In [0]:
# CREATE INITIAL CUSTOMER SCD2

customer_scd2 = (
    customer_base
    .withColumn(
        "hash_value",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("customer_id"), F.lit("")),
                F.coalesce(F.col("customer_name"), F.lit("")),
                F.coalesce(F.col("city"), F.lit("")),
                F.coalesce(F.col("segment"), F.lit("")),
                F.coalesce(F.col("gender"), F.lit("")),
                F.coalesce(
                    F.col("signup_date").cast("string"),
                    F.lit("")
                ),
                F.coalesce(F.col("status"), F.lit(""))
            ),
            256
        )
    )
    # Surrogate key
    # Unique for this historical version
    .withColumn(
        "customer_sk",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("customer_id"),
                F.col("signup_date").cast("string")
            ),
            256
        )
    )
    .withColumn(
        "effective_start_date",
        F.col("signup_date")
    )
    .withColumn(
        "effective_end_date",
        F.to_date(F.lit("9999-12-31"))
    )
    .withColumn(
        "is_current",
        F.lit(True)
    )

    .select(
        "customer_sk",
        "customer_id",
        "customer_name",
        "city",
        "segment",
        "gender",
        "signup_date",
        "status",
        "effective_start_date",
        "effective_end_date",
        "is_current",
        "hash_value"
    )
)

print("=" * 70)
print("INITIAL CUSTOMER SCD2")
print("=" * 70)

print("Rows:", customer_scd2.count())

print(
    "Distinct customer IDs:",
    customer_scd2.select("customer_id").distinct().count()
)

print(
    "Current records:",
    customer_scd2
    .filter(F.col("is_current") == True)
    .count()
)

customer_scd2.printSchema()

display(customer_scd2.limit(10))

INITIAL CUSTOMER SCD2
Rows: 2475
Distinct customer IDs: 2475
Current records: 2475
root
 |-- customer_sk: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- effective_start_date: date (nullable = true)
 |-- effective_end_date: date (nullable = true)
 |-- is_current: boolean (nullable = false)
 |-- hash_value: string (nullable = true)



customer_sk,customer_id,customer_name,city,segment,gender,signup_date,status,effective_start_date,effective_end_date,is_current,hash_value
99d447b1a3b86992599b86f697e20827da09d8d313aed4d5f747c0cf812744c0,C00001,Customer_1,Delhi,Regular,Other,2024-10-08,active,2024-10-08,9999-12-31,true,3a22b64ecfb1d3456ea7cf9889ce80c93a08ce67c95f9271bbf8dd639ccd89e0
169dc46dd375ba4a2c98c55aaacb1bb8a0159670d31362589ea122d9b318d42f,C00002,Customer_2,Bengaluru,Silver,Other,2024-04-14,active,2024-04-14,9999-12-31,true,babf34d9e869cf6059bb5dfad9f5f70a2881ecbae447773bd09d04168ccbaa5e
82c38b8c89b27af6137f82c777d30ee32b748119645642e1e8e4839d52e82f74,C00003,Customer_3,Gurugram,Platinum,M,2024-01-31,active,2024-01-31,9999-12-31,true,4f92d436c7d30c31a2f944032fcadb59833e8d8d157db12d326854462ef9a467
abb0861dc22dbc06b5836b015d02a5c4f629f5e651381a529a47c1a0d6518312,C00004,Customer_4,Bengaluru,Silver,Other,2025-09-08,active,2025-09-08,9999-12-31,true,85134a594082118d2ed19e77f61f2105d2a0d7375f7c4e581b578314c177ac93
d4d3c8225aedb5fb6c219fba4296cd551a0b58e720efbcc6170e8804b98a33aa,C00005,Customer_5,Ahmedabad,Silver,Other,2025-10-27,inactive,2025-10-27,9999-12-31,true,b497991157ba648bec6e810cd6aa0b350dc771062b49daf428a25bf34f88c488
ae2373055a06858ffa844bbeb94801a52aee58b2246a81d3df37ae66a3573964,C00006,Customer_6,Bengaluru,Platinum,Other,2024-10-11,active,2024-10-11,9999-12-31,true,9ddbad88c46425589656db9fd17e9c3f0baa1cd401df370b5a823349dba1bdef
4fd1e667dabb3c61279b89e2ad7aba16026f43b2220630cafc86a9d30b10d170,C00007,Customer_7,Mumbai,Platinum,F,2024-10-11,active,2024-10-11,9999-12-31,true,ca8b745649633106dd4dcd12d921902d1e121164097a25e8a532acd7d8d57fed
e096b10cc78566ca61078c61422b385a4143f1e8931d0ccaaba1a7d021484f23,C00008,Customer_8,Bengaluru,Gold,M,2024-04-04,inactive,2024-04-04,9999-12-31,true,e48dd536134cf0099f0157f35b77f60f329cf3e5d7319c77aa63fe094eb52956
82033d02d74c522e1c15a738a5a1ebf4b5e5a10dbd9fcd122db673a4909e0fc5,C00009,Customer_9,Delhi,Gold,F,2025-09-10,active,2025-09-10,9999-12-31,true,41270f53b2865895458c914280a6302f553d5855b7ad36eaffd6fb3760a1cdbe
2f88cb9e83fe51c99ed1552e07a1bbf7456fe628018b93ba95bd26b105d16b99,C00010,Customer_10,Jaipur,Platinum,Other,2024-05-07,inactive,2024-05-07,9999-12-31,true,05f9c51c86d699a375c64ea00a59f2c44a852e3f0dc2f9c45b1d82927f353176


In [0]:

#  VALIDATE INITIAL CUSTOMER SCD2
total_rows = customer_scd2.count()
# Distinct business keys
distinct_customer_ids = (
    customer_scd2
    .select("customer_id")
    .distinct()
    .count()
)

# Current rows
current_rows = (
    customer_scd2
    .filter(F.col("is_current") == True)
    .count()
)

# Historical/inactive rows
inactive_rows = (
    customer_scd2
    .filter(F.col("is_current") == False)
    .count()
)

# Open-ended rows
open_ended_rows = (
    customer_scd2
    .filter(
        F.col("effective_end_date") ==
        F.to_date(F.lit("9999-12-31"))
    )
    .count()
)

# Null surrogate keys
null_surrogate_keys = (
    customer_scd2
    .filter(F.col("customer_sk").isNull())
    .count()
)

# Null hashes
null_hash_values = (
    customer_scd2
    .filter(F.col("hash_value").isNull())
    .count()
)

# Duplicate surrogate keys
duplicate_surrogate_keys = (
    customer_scd2
    .groupBy("customer_sk")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Total rows:", total_rows)
print("Distinct customer IDs:", distinct_customer_ids)
print("Current rows:", current_rows)
print("Inactive rows:", inactive_rows)
print("Open-ended rows:", open_ended_rows)
print("Null surrogate keys:", null_surrogate_keys)
print("Null hash values:", null_hash_values)
print("Duplicate surrogate keys:", duplicate_surrogate_keys)

print("\nSample:")
display(customer_scd2.limit(10))

Total rows: 2475
Distinct customer IDs: 2475
Current rows: 2475
Inactive rows: 0
Open-ended rows: 2475
Null surrogate keys: 0
Null hash values: 0
Duplicate surrogate keys: 0

Sample:


customer_sk,customer_id,customer_name,city,segment,gender,signup_date,status,effective_start_date,effective_end_date,is_current,hash_value
99d447b1a3b86992599b86f697e20827da09d8d313aed4d5f747c0cf812744c0,C00001,Customer_1,Delhi,Regular,Other,2024-10-08,active,2024-10-08,9999-12-31,true,3a22b64ecfb1d3456ea7cf9889ce80c93a08ce67c95f9271bbf8dd639ccd89e0
169dc46dd375ba4a2c98c55aaacb1bb8a0159670d31362589ea122d9b318d42f,C00002,Customer_2,Bengaluru,Silver,Other,2024-04-14,active,2024-04-14,9999-12-31,true,babf34d9e869cf6059bb5dfad9f5f70a2881ecbae447773bd09d04168ccbaa5e
82c38b8c89b27af6137f82c777d30ee32b748119645642e1e8e4839d52e82f74,C00003,Customer_3,Gurugram,Platinum,M,2024-01-31,active,2024-01-31,9999-12-31,true,4f92d436c7d30c31a2f944032fcadb59833e8d8d157db12d326854462ef9a467
abb0861dc22dbc06b5836b015d02a5c4f629f5e651381a529a47c1a0d6518312,C00004,Customer_4,Bengaluru,Silver,Other,2025-09-08,active,2025-09-08,9999-12-31,true,85134a594082118d2ed19e77f61f2105d2a0d7375f7c4e581b578314c177ac93
d4d3c8225aedb5fb6c219fba4296cd551a0b58e720efbcc6170e8804b98a33aa,C00005,Customer_5,Ahmedabad,Silver,Other,2025-10-27,inactive,2025-10-27,9999-12-31,true,b497991157ba648bec6e810cd6aa0b350dc771062b49daf428a25bf34f88c488
ae2373055a06858ffa844bbeb94801a52aee58b2246a81d3df37ae66a3573964,C00006,Customer_6,Bengaluru,Platinum,Other,2024-10-11,active,2024-10-11,9999-12-31,true,9ddbad88c46425589656db9fd17e9c3f0baa1cd401df370b5a823349dba1bdef
4fd1e667dabb3c61279b89e2ad7aba16026f43b2220630cafc86a9d30b10d170,C00007,Customer_7,Mumbai,Platinum,F,2024-10-11,active,2024-10-11,9999-12-31,true,ca8b745649633106dd4dcd12d921902d1e121164097a25e8a532acd7d8d57fed
e096b10cc78566ca61078c61422b385a4143f1e8931d0ccaaba1a7d021484f23,C00008,Customer_8,Bengaluru,Gold,M,2024-04-04,inactive,2024-04-04,9999-12-31,true,e48dd536134cf0099f0157f35b77f60f329cf3e5d7319c77aa63fe094eb52956
82033d02d74c522e1c15a738a5a1ebf4b5e5a10dbd9fcd122db673a4909e0fc5,C00009,Customer_9,Delhi,Gold,F,2025-09-10,active,2025-09-10,9999-12-31,true,41270f53b2865895458c914280a6302f553d5855b7ad36eaffd6fb3760a1cdbe
2f88cb9e83fe51c99ed1552e07a1bbf7456fe628018b93ba95bd26b105d16b99,C00010,Customer_10,Jaipur,Platinum,Other,2024-05-07,inactive,2024-05-07,9999-12-31,true,05f9c51c86d699a375c64ea00a59f2c44a852e3f0dc2f9c45b1d82927f353176


In [0]:
#WRITE INITIAL CUSTOMER SCD2

CUSTOMER_SCD2_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_scd2"
)

(
    customer_scd2
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(CUSTOMER_SCD2_TABLE)
)

print("Table:", CUSTOMER_SCD2_TABLE)

display(
    spark.sql(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT customer_id) AS distinct_customers,
            SUM(CASE WHEN is_current = true THEN 1 ELSE 0 END)
                AS current_rows
        FROM {CUSTOMER_SCD2_TABLE}
    """)
)

Table: retail_demo.silver.dim_customer_scd2


total_rows,distinct_customers,current_rows
2475,2475,2475


In [0]:
# PREPARE CUSTOMER CDC

customer_cdc = (
    spark.table(
        f"{CATALOG}.{RAW_SCHEMA}.bronze_customers_incremental"
    )
    .select(
        "customer_id",
        "customer_name",
        "city",
        "segment",
        "gender",
        "signup_date",
        "status",
        "effective_date",
        "operation",
        "ingest_ts"
    )

    # Clean strings
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("customer_name", F.trim(F.col("customer_name")))
    .withColumn("city", F.trim(F.col("city")))
    .withColumn("segment", F.trim(F.col("segment")))
    .withColumn("gender", F.trim(F.col("gender")))
    .withColumn("status", F.trim(F.col("status")))
    .withColumn("operation", F.trim(F.col("operation")))

    # Parse dates
    .withColumn(
        "signup_date",
        F.coalesce(
            F.expr(
                "try_to_date(trim(signup_date), 'yyyy-MM-dd')"
            ),
            F.expr(
                "try_to_date(trim(signup_date), 'dd-MM-yyyy')"
            )
        )
    )
    .withColumn(
        "effective_date",
        F.coalesce(
            F.expr(
                "try_to_date(trim(effective_date), 'yyyy-MM-dd')"
            ),
            F.expr(
                "try_to_date(trim(effective_date), 'dd-MM-yyyy')"
            )
        )
    )

    # Use effective date as change start date
    .withColumn(
        "effective_start_date",
        F.coalesce(
            F.col("effective_date"),
            F.col("signup_date")
        )
    )
)

print("CDC rows:", customer_cdc.count())

print(
    "Distinct customer IDs:",
    customer_cdc.select("customer_id").distinct().count()
)

print("\nOperation counts:")

display(
    customer_cdc
    .groupBy("operation")
    .count()
    .orderBy("operation")
)

print("\nSample CDC records:")

display(
    customer_cdc.limit(10)
)

CDC rows: 630
Distinct customer IDs: 582

Operation counts:


operation,count
INSERT,251
UPDATE,379



Sample CDC records:


customer_id,customer_name,city,segment,gender,signup_date,status,effective_date,operation,ingest_ts,effective_start_date
C00606,Customer_606,Kolkata,Silver,Other,2025-10-09,active,null,UPDATE,2026-08-11T18:10:49.962Z,2025-10-09
C00527,Customer_527,Bengaluru,Regular,M,2024-04-10,inactive,2026-04-25,UPDATE,2026-08-11T18:10:49.962Z,2026-04-25
C00660,Customer_660,Jaipur,Regular,F,2024-10-16,active,2026-04-25,UPDATE,2026-08-11T18:10:49.962Z,2026-04-25
C01018,Customer_1018,Jaipur,Platinum,F,2025-04-06,active,2026-04-25,UPDATE,2026-08-11T18:10:49.962Z,2026-04-25
C00781,Customer_781,Pune,Regular,F,2025-12-01,active,2026-04-25,UPDATE,2026-08-11T18:10:49.962Z,2026-04-25
C02068,Customer_2068,Delhi,Silver,M,2024-10-14,active,2026-04-25,UPDATE,2026-08-11T18:10:49.962Z,2026-04-25
C02152,Customer_2152,Hyderabad,Gold,M,2024-05-30,active,2026-04-25,UPDATE,2026-08-11T18:10:49.962Z,2026-04-25
C01200,Customer_1200,Delhi,Gold,F,2024-03-30,active,2026-04-25,UPDATE,2026-08-11T18:10:49.962Z,2026-04-25
C00959,Customer_959,Ahmedabad,Platinum,F,2025-09-02,active,2026-04-25,UPDATE,2026-08-11T18:10:49.962Z,2026-04-25
C00439,Customer_439,Delhi,Platinum,Other,2025-08-10,active,2026-04-25,UPDATE,2026-08-11T18:10:49.962Z,2026-04-25


In [0]:
# CUSTOMER CDC HASH & CHANGE DETECTION
# Create hash for incoming CDC state

customer_cdc_hashed = (
    customer_cdc

    .withColumn(
        "hash_value",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("customer_id"), F.lit("")),
                F.coalesce(F.col("customer_name"), F.lit("")),
                F.coalesce(F.col("city"), F.lit("")),
                F.coalesce(F.col("segment"), F.lit("")),
                F.coalesce(F.col("gender"), F.lit("")),
                F.coalesce(
                    F.col("signup_date").cast("string"),
                    F.lit("")
                ),
                F.coalesce(F.col("status"), F.lit(""))
            ),
            256
        )
    )
)
# Compare incoming CDC against the CURRENT SCD2 records


current_customers = (
    spark.table(
        f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_scd2"
    )
    .filter(F.col("is_current") == True)
    .select(
        "customer_id",
        F.col("hash_value").alias("existing_hash"),
        F.col("customer_sk").alias("existing_customer_sk")
    )
)

customer_cdc_changes = (
    customer_cdc_hashed
    .join(
        current_customers,
        on="customer_id",
        how="left"
    )
    .withColumn(
        "change_type",
        F.when(
            F.col("existing_customer_sk").isNull(),
            F.lit("INSERT")
        )
        .when(
            F.col("hash_value") != F.col("existing_hash"),
            F.lit("UPDATE")
        )
        .otherwise(
            F.lit("NO_CHANGE")
        )
    )
)
print(
    "Total CDC records:",
    customer_cdc_changes.count()
)

print("\nChange classification:")

display(
    customer_cdc_changes
    .groupBy("change_type")
    .count()
    .orderBy("change_type")
)

Total CDC records: 630

Change classification:


change_type,count
INSERT,255
NO_CHANGE,9
UPDATE,366


In [0]:
# PREPARE CUSTOMER SCD2 CHANGES


customer_changes = (
    customer_cdc_changes
    .filter(
        F.col("change_type").isin(["INSERT", "UPDATE"])
    )
    .select(
        "customer_id",
        "customer_name",
        "city",
        "segment",
        "gender",
        "signup_date",
        "status",
        "effective_start_date",
        "operation",
        "hash_value",
        "change_type"
    )
)
print(
    "Total changes:",
    customer_changes.count()
)

print("\nChanges by type:")

display(
    customer_changes
    .groupBy("change_type")
    .count()
    .orderBy("change_type")
)

print("\nSample changes:")

display(
    customer_changes.limit(10)
)

Total changes: 621

Changes by type:


change_type,count
INSERT,255
UPDATE,366



Sample changes:


customer_id,customer_name,city,segment,gender,signup_date,status,effective_start_date,operation,hash_value,change_type
C00527,Customer_527,Bengaluru,Regular,M,2024-04-10,inactive,2026-04-25,UPDATE,793d36120dfb72d4d6fc5dc7702087d2aaa35abf2593cff7748a0af5e1489672,UPDATE
C00660,Customer_660,Jaipur,Regular,F,2024-10-16,active,2026-04-25,UPDATE,cd4013dfcbff652e2b7232691333d7e186849988d83cb34a813717a495bb361b,UPDATE
C01018,Customer_1018,Jaipur,Platinum,F,2025-04-06,active,2026-04-25,UPDATE,40044214e82a5de352006253ef5bd04b897f803b9bc73ceec0a79f8659ed3679,UPDATE
C00781,Customer_781,Pune,Regular,F,2025-12-01,active,2026-04-25,UPDATE,5dcfcc3e5ce13525ac43af5039da5ffe88e6d29a32c174d2a405500f88d02d82,UPDATE
C02068,Customer_2068,Delhi,Silver,M,2024-10-14,active,2026-04-25,UPDATE,f90ac723127d2d9361bf29f8b1e65e8c2176ba4b2a25739016ac116cb1ee5371,UPDATE
C02152,Customer_2152,Hyderabad,Gold,M,2024-05-30,active,2026-04-25,UPDATE,52a1996656f5a9c3150abf69d47e27e40f6ac903ae8f3a66078604fc2d100a51,UPDATE
C01200,Customer_1200,Delhi,Gold,F,2024-03-30,active,2026-04-25,UPDATE,b6efe072412dbcdcbb1c4470c26753db01b1ed19621f38e32bf212f77d6debac,UPDATE
C00959,Customer_959,Ahmedabad,Platinum,F,2025-09-02,active,2026-04-25,UPDATE,a3ae21ba83ff5b375114f86cdb138260d3effd61f67895a59de96ca2b5e1ce6b,UPDATE
C00439,Customer_439,Delhi,Platinum,Other,2025-08-10,active,2026-04-25,UPDATE,9726f1f4c2dd8dd0ffdc57587af3dfb4fc61a2e21de90bf3a8c9ee86477dc45b,UPDATE
C01595,Customer_1595,Hyderabad,Platinum,F,2026-01-24,inactive,2026-04-25,UPDATE,5706b1fb8d42d897a61a92bdf4ce025318ba6aa455e72df75866b61f59c17ed0,UPDATE


In [0]:
# EXPIRE CHANGED CUSTOMER RECORDS

from delta.tables import DeltaTable

customer_dim = DeltaTable.forName(
    spark,
    f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_scd2"
)

customer_updates = (
    customer_changes
    .filter(F.col("change_type") == "UPDATE")
    .select(
        "customer_id",
        "effective_start_date",
        "hash_value"
    )
    .dropDuplicates(["customer_id"])
)
print(
    "Customer updates to process:",
    customer_updates.count()
)

(
    customer_dim.alias("t")
    .merge(
        customer_updates.alias("s"),
        """
        t.customer_id = s.customer_id
        AND t.is_current = true
        """
    )
    .whenMatchedUpdate(
        condition="t.hash_value <> s.hash_value",
        set={
            "effective_end_date": "date_sub(s.effective_start_date, 1)",
            "is_current": "false"
        }
    )
    .execute()
)

print("Changed customer records expired successfully.")

Customer updates to process: 332
Changed customer records expired successfully.


In [0]:
# RESET CUSTOMER SCD2 TO INITIAL STATE


CUSTOMER_SCD2_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_scd2"
)

(
    customer_scd2
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(CUSTOMER_SCD2_TABLE)
)
display(
    spark.sql(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT customer_id) AS distinct_customers,
            SUM(
                CASE WHEN is_current = true THEN 1 ELSE 0 END
            ) AS current_rows,
            SUM(
                CASE WHEN is_current = false THEN 1 ELSE 0 END
            ) AS inactive_rows
        FROM {CUSTOMER_SCD2_TABLE}
    """)
)

total_rows,distinct_customers,current_rows,inactive_rows
2475,2475,2475,0


In [0]:
# FREEZE CUSTOMER CDC CHANGE SET


customer_cdc_rebuild = (
    spark.table(
        f"{CATALOG}.{RAW_SCHEMA}.bronze_customers_incremental"
    )
    .select(
        "customer_id",
        "customer_name",
        "city",
        "segment",
        "gender",
        "signup_date",
        "status",
        "effective_date",
        "operation",
        "ingest_ts"
    )
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("customer_name", F.trim(F.col("customer_name")))
    .withColumn("city", F.trim(F.col("city")))
    .withColumn("segment", F.trim(F.col("segment")))
    .withColumn("gender", F.trim(F.col("gender")))
    .withColumn("status", F.trim(F.col("status")))
    .withColumn("operation", F.trim(F.col("operation")))
    .withColumn(
        "signup_date",
        F.coalesce(
            F.expr("try_to_date(trim(signup_date), 'yyyy-MM-dd')"),
            F.expr("try_to_date(trim(signup_date), 'dd-MM-yyyy')")
        )
    )
    .withColumn(
        "effective_date",
        F.coalesce(
            F.expr("try_to_date(trim(effective_date), 'yyyy-MM-dd')"),
            F.expr("try_to_date(trim(effective_date), 'dd-MM-yyyy')")
        )
    )
    .withColumn(
        "effective_start_date",
        F.coalesce(
            F.col("effective_date"),
            F.col("signup_date")
        )
    )
    .withColumn(
        "hash_value",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("customer_id"), F.lit("")),
                F.coalesce(F.col("customer_name"), F.lit("")),
                F.coalesce(F.col("city"), F.lit("")),
                F.coalesce(F.col("segment"), F.lit("")),
                F.coalesce(F.col("gender"), F.lit("")),
                F.coalesce(
                    F.col("signup_date").cast("string"),
                    F.lit("")
                ),
                F.coalesce(F.col("status"), F.lit(""))
            ),
            256
        )
    )
)

current_customer_dim = (
    spark.table(
        f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_scd2"
    )
    .filter(F.col("is_current") == True)
    .select(
        "customer_id",
        F.col("hash_value").alias("existing_hash")
    )
)

# Build the complete classification
customer_changes_frozen = (
    customer_cdc_rebuild
    .join(
        current_customer_dim,
        "customer_id",
        "left"
    )
    .withColumn(
        "change_type",
        F.when(
            F.col("existing_hash").isNull(),
            F.lit("INSERT")
        )
        .when(
            F.col("hash_value") != F.col("existing_hash"),
            F.lit("UPDATE")
        )
        .otherwise(
            F.lit("NO_CHANGE")
        )
    )
    .select(
        "customer_id",
        "customer_name",
        "city",
        "segment",
        "gender",
        "signup_date",
        "status",
        "effective_start_date",
        "operation",
        "hash_value",
        "change_type"
    )
)

# Force the DataFrame to execute before we touch the target
change_counts = (
    customer_changes_frozen
    .groupBy("change_type")
    .count()
    .collect()
)
for row in change_counts:
    print(f"{row['change_type']}: {row['count']}")

print("Total CDC records:", customer_changes_frozen.count())

NO_CHANGE: 9
UPDATE: 366
INSERT: 255
Total CDC records: 630


In [0]:
#  FINAL DETERMINISTIC CUSTOMER CDC SET


# Get ingestion timestamps from the raw CDC preparation
cdc_ingest_info = (
    customer_cdc_rebuild
    .select(
        "customer_id",
        "effective_start_date",
        "hash_value",
        "ingest_ts"
    )
)

# Add ingest_ts to the already-frozen CDC classification
customer_cdc_deterministic = (
    customer_changes_frozen
    .join(
        cdc_ingest_info,
        on=[
            "customer_id",
            "effective_start_date",
            "hash_value"
        ],
        how="left"
    )
    .filter(
        F.col("change_type").isin(["INSERT", "UPDATE"])
    )
    .withColumn(
        "rn_same_day",
        F.row_number().over(
            Window
            .partitionBy(
                "customer_id",
                "effective_start_date"
            )
            .orderBy(
                F.col("ingest_ts").desc()
            )
        )
    )
    .filter(F.col("rn_same_day") == 1)
    .drop("rn_same_day")
)

print("=" * 70)
print("CUSTOMER CDC — FINAL DETERMINISTIC CHANGE SET")
print("=" * 70)

print(
    "Original change records:",
    customer_changes_frozen
    .filter(
        F.col("change_type").isin(["INSERT", "UPDATE"])
    )
    .count()
)

print(
    "After same-day deduplication:",
    customer_cdc_deterministic.count()
)

print(
    "Customers with multiple change dates:",
    customer_cdc_deterministic
    .groupBy("customer_id")
    .agg(
        F.countDistinct("effective_start_date")
        .alias("change_dates")
    )
    .filter(F.col("change_dates") > 1)
    .count()
)

display(
    customer_cdc_deterministic
    .select(
        "customer_id",
        "change_type",
        "effective_start_date",
        "ingest_ts",
        "operation",
        "hash_value"
    )
    .orderBy(
        "customer_id",
        "effective_start_date",
        "ingest_ts"
    )
    .limit(50)
)

CUSTOMER CDC — FINAL DETERMINISTIC CHANGE SET
Original change records: 621
After same-day deduplication: 592
Customers with multiple change dates: 16


customer_id,change_type,effective_start_date,ingest_ts,operation,hash_value
C00015,UPDATE,2026-04-25,2026-08-11T18:10:49.962Z,UPDATE,aa1d95ca5843291c3d46478ffe47cc06a50cc12ab500d59fb48d7963c4f61301
C00015,UPDATE,2026-04-26,2026-08-11T18:10:49.962Z,UPDATE,68fd5e8cf35bd79d986dc9de16e64bc6ef665f1bc438f0cb6f0699fa650d2af0
C00020,UPDATE,2026-04-26,2026-08-11T18:10:49.962Z,UPDATE,96ffb59524116181e7af8daeeaf53e5332040d52e2671cb0da04dcba1d9e0e58
C00022,UPDATE,2026-04-25,2026-08-11T18:10:49.962Z,UPDATE,7e4f6097db1913b3a2e0703821dd856efea6fb460702d53bbfffea290b53e619
C00039,UPDATE,2026-04-24,2026-08-11T18:10:49.962Z,UPDATE,7e69da7533dff6231e83f3180d3ad0254e7b5fc32e0ede11d7e31bb071e781e9
C00044,UPDATE,2026-04-24,2026-08-11T18:10:49.962Z,UPDATE,46198a41dc33f923f5f87507cb90146eb9259afcbe3d94940083e518be7fe74d
C00066,INSERT,2026-04-24,2026-08-11T18:10:49.962Z,UPDATE,1a66e1802ff87a177826d51fbe8bbe7ca5ea1ced6ccb1ef1591a0fcd6517141e
C00067,UPDATE,2026-04-25,2026-08-11T18:10:49.962Z,UPDATE,cd536cad888047b8ff9608e0d96b6a542a943d086d5f94adca3f74f391043b9f
C00068,UPDATE,2026-04-25,2026-08-11T18:10:49.962Z,UPDATE,e1a4d3cb8f329fc541b66dc9fe6033d7f4304e94fe226c91da056157c4530e23
C00081,UPDATE,2026-04-24,2026-08-11T18:10:49.962Z,UPDATE,54a2e9bef08e88552feddaef4e16194c9105663ea42b2b6231cc7e9e083673ef


In [0]:
#  REVIEW CUSTOMER CDC PROCESSING DATES

cdc_date_summary = (
    customer_cdc_deterministic
    .groupBy("effective_start_date")
    .agg(
        F.count("*").alias("records"),
        F.countDistinct("customer_id").alias("customers")
    )
    .orderBy("effective_start_date")
)
display(cdc_date_summary)
print(
    "Total CDC records to process:",
    customer_cdc_deterministic.count()
)

effective_start_date,records,customers
2026-04-24,196,196
2026-04-25,196,196
2026-04-26,200,200


Total CDC records to process: 592


In [0]:
#  APPLY CUSTOMER SCD2 CHRONOLOGICALLY

CUSTOMER_SCD2_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_scd2"
)

customer_dim = DeltaTable.forName(
    spark,
    CUSTOMER_SCD2_TABLE
)

cdc_dates = [
    row["effective_start_date"]
    for row in (
        customer_cdc_deterministic
        .select("effective_start_date")
        .distinct()
        .orderBy("effective_start_date")
        .collect()
    )
]
for change_date in cdc_dates:

    print(f"\nProcessing CDC date: {change_date}")
    daily_cdc = (
        customer_cdc_deterministic
        .filter(
            F.col("effective_start_date") == F.lit(change_date)
        )
        .select(
            "customer_id",
            "customer_name",
            "city",
            "segment",
            "gender",
            "signup_date",
            "status",
            "effective_start_date",
            "hash_value"
        )
        .withColumn(
            "customer_sk",
            F.sha2(
                F.concat_ws(
                    "||",
                    F.col("customer_id"),
                    F.col("effective_start_date").cast("string"),
                    F.col("hash_value")
                ),
                256
            )
        )
        .withColumn(
            "effective_end_date",
            F.to_date(F.lit("9999-12-31"))
        )
        .withColumn(
            "is_current",
            F.lit(True)
        )
    )

    daily_count = daily_cdc.count()

    print("  Incoming records:", daily_count)

    (
        customer_dim.alias("t")
        .merge(
            daily_cdc.alias("s"),
            """
            t.customer_id = s.customer_id
            AND t.is_current = true
            """
        )
        .whenMatchedUpdate(
            condition="""
                t.hash_value <> s.hash_value
            """,
            set={
                "effective_end_date":
                    "date_sub(s.effective_start_date, 1)",
                "is_current":
                    "false"
            }
        )
        .execute()
    )

    (
        customer_dim.alias("t")
        .merge(
            daily_cdc.alias("s"),
            """
            t.customer_id = s.customer_id
            AND t.is_current = true
            AND t.hash_value = s.hash_value
            """
        )
        .whenNotMatchedInsert(
            values={
                "customer_sk": "s.customer_sk",
                "customer_id": "s.customer_id",
                "customer_name": "s.customer_name",
                "city": "s.city",
                "segment": "s.segment",
                "gender": "s.gender",
                "signup_date": "s.signup_date",
                "status": "s.status",
                "effective_start_date":
                    "s.effective_start_date",
                "effective_end_date":
                    "s.effective_end_date",
                "is_current": "s.is_current",
                "hash_value": "s.hash_value"
            }
        )
        .execute()
    )

    print("  Completed:", change_date)

print("\nCustomer SCD2 chronological processing completed.")


Processing CDC date: 2026-04-24
  Incoming records: 196
  Completed: 2026-04-24

Processing CDC date: 2026-04-25
  Incoming records: 197
  Completed: 2026-04-25

Processing CDC date: 2026-04-26
  Incoming records: 200
  Completed: 2026-04-26

Customer SCD2 chronological processing completed.


In [0]:
#  MATERIALIZE FINAL CUSTOMER CDC IN MEMORY

initial_customer_lookup = (
    customer_scd2
    .select(
        "customer_id",
        F.col("hash_value").alias("existing_hash")
    )
)

cdc_classified_once = (
    customer_cdc_rebuild
    .join(
        initial_customer_lookup,
        on="customer_id",
        how="left"
    )
    .withColumn(
        "change_type",
        F.when(
            F.col("existing_hash").isNull(),
            F.lit("INSERT")
        )
        .when(
            F.col("hash_value") != F.col("existing_hash"),
            F.lit("UPDATE")
        )
        .otherwise(
            F.lit("NO_CHANGE")
        )
    )
    .filter(
        F.col("change_type").isin(["INSERT", "UPDATE"])
    )
    .select(
        "customer_id",
        "customer_name",
        "city",
        "segment",
        "gender",
        "signup_date",
        "status",
        "effective_start_date",
        "operation",
        "hash_value",
        "change_type",
        "ingest_ts"
    )
)

# Collect the complete change set BEFORE touching the target table.
frozen_rows = cdc_classified_once.collect()
print("Frozen change records:", len(frozen_rows))
# Recreate as a Spark DataFrame from the collected rows.
customer_cdc_true_frozen = spark.createDataFrame(
    frozen_rows,
    schema=cdc_classified_once.schema
)
print("\nChange classification:")
display(
    customer_cdc_true_frozen
    .groupBy("change_type")
    .count()
    .orderBy("change_type")
)


Frozen change records: 621

Change classification:


change_type,count
INSERT,255
UPDATE,366


In [0]:
# RESET CUSTOMER DIMENSION BEFORE FINAL SCD2 APPLY

CUSTOMER_SCD2_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_scd2"
)

(
    customer_scd2
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(CUSTOMER_SCD2_TABLE)
)
display(
    spark.sql(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT customer_id) AS distinct_customers,
            SUM(
                CASE WHEN is_current = true THEN 1 ELSE 0 END
            ) AS current_rows,
            SUM(
                CASE WHEN is_current = false THEN 1 ELSE 0 END
            ) AS historical_rows
        FROM {CUSTOMER_SCD2_TABLE}
    """)
)

total_rows,distinct_customers,current_rows,historical_rows
2475,2475,2475,0


In [0]:
# FINAL CUSTOMER CDC SET FOR SCD2
customer_cdc_apply = (
    customer_cdc_true_frozen

    # Keep only actual INSERT / UPDATE records
    .filter(
        F.col("change_type").isin(["INSERT", "UPDATE"])
    )
    .withColumn(
        "rn",
        F.row_number().over(
            Window
            .partitionBy(
                "customer_id",
                "effective_start_date"
            )
            .orderBy(
                F.col("ingest_ts").desc()
            )
        )
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)
print(
    "Original changes:",
    customer_cdc_true_frozen.count()
)

print(
    "After same-day deduplication:",
    customer_cdc_apply.count()
)

print("\nRecords by effective date:")

display(
    customer_cdc_apply
    .groupBy("effective_start_date")
    .count()
    .orderBy("effective_start_date")
)

Original changes: 621
After same-day deduplication: 592

Records by effective date:


effective_start_date,count
2026-04-24,196
2026-04-25,196
2026-04-26,200


In [0]:
#  CUSTOMER SCD2 APPLY

CUSTOMER_SCD2_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_scd2"
)

customer_dim = DeltaTable.forName(
    spark,
    CUSTOMER_SCD2_TABLE
)

# Get the three CDC dates from the frozen/deduplicated set
cdc_dates = [
    row["effective_start_date"]
    for row in (
        customer_cdc_apply
        .select("effective_start_date")
        .distinct()
        .orderBy("effective_start_date")
        .collect()
    )
]
for change_date in cdc_dates:

    print(f"\nProcessing: {change_date}")

    daily_cdc = (
        customer_cdc_apply
        .filter(
            F.col("effective_start_date") == F.lit(change_date)
        )
        .select(
            "customer_id",
            "customer_name",
            "city",
            "segment",
            "gender",
            "signup_date",
            "status",
            "effective_start_date",
            "hash_value",
            "change_type"
        )
        .withColumn(
            "customer_sk",
            F.sha2(
                F.concat_ws(
                    "||",
                    F.col("customer_id"),
                    F.col("effective_start_date").cast("string"),
                    F.col("hash_value")
                ),
                256
            )
        )
        .withColumn(
            "effective_end_date",
            F.to_date(F.lit("9999-12-31"))
        )
        .withColumn(
            "is_current",
            F.lit(True)
        )
    )

    print("Incoming:", daily_cdc.count())

    daily_updates = (
        daily_cdc
        .filter(F.col("change_type") == "UPDATE")
    )

    (
        customer_dim.alias("t")
        .merge(
            daily_updates.alias("s"),
            """
            t.customer_id = s.customer_id
            AND t.is_current = true
            """
        )
        .whenMatchedUpdate(
            condition="t.hash_value <> s.hash_value",
            set={
                "effective_end_date":
                    "date_sub(s.effective_start_date, 1)",
                "is_current":
                    "false"
            }
        )
        .execute()
    )

    (
        customer_dim.alias("t")
        .merge(
            daily_cdc.alias("s"),
            """
            t.customer_id = s.customer_id
            AND t.is_current = true
            AND t.hash_value = s.hash_value
            """
        )
        .whenNotMatchedInsert(
            values={
                "customer_sk": "s.customer_sk",
                "customer_id": "s.customer_id",
                "customer_name": "s.customer_name",
                "city": "s.city",
                "segment": "s.segment",
                "gender": "s.gender",
                "signup_date": "s.signup_date",
                "status": "s.status",
                "effective_start_date":
                    "s.effective_start_date",
                "effective_end_date":
                    "s.effective_end_date",
                "is_current":
                    "s.is_current",
                "hash_value":
                    "s.hash_value"
            }
        )
        .execute()
    )
    print("Completed:", change_date)

print("\nCustomer SCD2 apply completed successfully.")


Processing: 2026-04-24
Incoming: 196
Completed: 2026-04-24

Processing: 2026-04-25
Incoming: 196
Completed: 2026-04-25

Processing: 2026-04-26
Incoming: 200
Completed: 2026-04-26

Customer SCD2 apply completed successfully.


In [0]:
#  FINAL CUSTOMER SCD2 VALIDATION


customer_dim_final = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_scd2"
)
total_rows = customer_dim_final.count()
distinct_customers = (
    customer_dim_final
    .select("customer_id")
    .distinct()
    .count()
)
current_rows = (
    customer_dim_final
    .filter(F.col("is_current") == True)
    .count()
)
historical_rows = (
    customer_dim_final
    .filter(F.col("is_current") == False)
    .count()
)
open_ended_rows = (
    customer_dim_final
    .filter(
        F.col("effective_end_date") ==
        F.to_date(F.lit("9999-12-31"))
    )
    .count()
)
null_surrogate_keys = (
    customer_dim_final
    .filter(F.col("customer_sk").isNull())
    .count()
)
duplicate_surrogate_keys = (
    customer_dim_final
    .groupBy("customer_sk")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

multiple_current_rows = (
    customer_dim_final
    .groupBy("customer_id")
    .agg(
        F.sum(
            F.when(F.col("is_current") == True, 1)
             .otherwise(0)
        ).alias("current_count")
    )
    .filter(F.col("current_count") > 1)
    .count()
)

print("Total dimension rows:", total_rows)
print("Distinct customer IDs:", distinct_customers)
print("Current rows:", current_rows)
print("Historical rows:", historical_rows)
print("Open-ended rows:", open_ended_rows)
print("Null surrogate keys:", null_surrogate_keys)
print("Duplicate surrogate keys:", duplicate_surrogate_keys)
print("Customers with multiple current rows:", multiple_current_rows)
display(
    customer_dim_final
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
    .limit(20)
)

Total dimension rows: 3067
Distinct customer IDs: 2719
Current rows: 2719
Historical rows: 348
Open-ended rows: 2719
Null surrogate keys: 0
Duplicate surrogate keys: 0
Customers with multiple current rows: 0


customer_id,count
C00577,3
C00538,3
C01154,3
C00948,3
C00015,3
C00660,3
C01129,3
C02395,3
C00367,3
C00084,3


In [0]:
# CUSTOMER SCD2 TEMPORAL VALIDATION


customer_dim_check = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_scd2"
)

versioned_customers = (
    customer_dim_check
    .withColumn(
        "next_start_date",
        F.lead("effective_start_date").over(
            Window
            .partitionBy("customer_id")
            .orderBy("effective_start_date")
        )
    )
)

invalid_ranges = (
    versioned_customers
    .filter(
        F.col("next_start_date").isNotNull()
        & (
            F.col("effective_end_date")
            >= F.col("next_start_date")
        )
    )
)

print(
    "Invalid/overlapping date ranges:",
    invalid_ranges.count()
)

version_counts = (
    customer_dim_check
    .groupBy("customer_id")
    .count()
)

print(
    "Customers with historical versions:",
    version_counts
    .filter(F.col("count") > 1)
    .count()
)

current_check = (
    customer_dim_check
    .groupBy("customer_id")
    .agg(
        F.sum(
            F.when(F.col("is_current") == True, 1)
             .otherwise(0)
        ).alias("current_count")
    )
)

print(
    "Customers with incorrect current-row count:",
    current_check
    .filter(F.col("current_count") != 1)
    .count()
)

print("\nSample historical customers:")

display(
    customer_dim_check
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
    .limit(10)
)

Invalid/overlapping date ranges: 0
Customers with historical versions: 332
Customers with incorrect current-row count: 0

Sample historical customers:


customer_id,count
C02395,3
C00538,3
C01129,3
C00084,3
C00929,3
C00660,3
C00552,3
C02309,3
C01154,3
C00938,3


In [0]:
# LOAD HISTORICAL PRODUCTS

products_batch = spark.table(
    f"{CATALOG}.{RAW_SCHEMA}.products_bronze"
)
print("Bronze batch rows:", products_batch.count())
products_batch.printSchema()
display(
    products_batch.limit(10)
)

Bronze batch rows: 830
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- status: string (nullable = true)
 |-- created_date: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_type: string (nullable = true)



product_id,product_name,category,brand,unit_price,status,created_date,_ingested_at,_source_file,_source_type
P00001,T-Shirt 1,Fashion,BrandC,unknown,discontinued,2025-02-15,2026-08-12T17:22:04.533Z,products_batch.csv,batch
P00002,Bedsheet 2,Home,BrandC,unknown,active,2024-04-20,2026-08-12T17:22:04.533Z,products_batch.csv,batch
P00003,Bedsheet 3,Home,BrandD,30753.26,active,2024-10-23,2026-08-12T17:22:04.533Z,products_batch.csv,batch
P00004,Oil 4,Grocery,BrandB,68176.44,active,2024-03-07,2026-08-12T17:22:04.533Z,products_batch.csv,batch
P00005,Oil 5,null,BrandC,51796.39,active,2024-12-12,2026-08-12T17:22:04.533Z,products_batch.csv,batch
P00006,Chair 6,Home,BrandC,10740.11,discontinued,2025-06-17,2026-08-12T17:22:04.533Z,products_batch.csv,batch
P00007,Jeans 7,Fashion,BrandC,26020.02,active,2024-04-19,2026-08-12T17:22:04.533Z,products_batch.csv,batch
P00008,Chair 8,Home,BrandB,8710.16,active,2023-03-23,2026-08-12T17:22:04.533Z,products_batch.csv,batch
P00009,Cream 9,Beauty,BrandA,71925.36,discontinued,2023-02-22,2026-08-12T17:22:04.533Z,products_batch.csv,batch
P00010,Chair 10,Home,BrandD,25422.83,discontinued,2025-12-02,2026-08-12T17:22:04.533Z,products_batch.csv,batch


In [0]:
#  PREPARE HISTORICAL PRODUCT BASE


product_base = (
    products_batch
    .select(
        "product_id",
        "product_name",
        "category",
        "brand",
        "unit_price",
        "status",
        "created_date",
        "_ingested_at"
    )
    .withColumn("product_id", F.trim(F.col("product_id")))
    .withColumn("product_name", F.trim(F.col("product_name")))
    .withColumn("category", F.trim(F.col("category")))
    .withColumn("brand", F.trim(F.col("brand")))
    .withColumn("unit_price_raw", F.trim(F.col("unit_price")))
    .withColumn("status", F.trim(F.col("status")))
    .withColumn(
        "category",
        F.when(
            F.col("category").isNull() |
            (F.col("category") == ""),
            F.lit("Unknown")
        ).otherwise(F.col("category"))
    )
    .withColumn(
        "brand",
        F.when(
            F.col("brand").isNull() |
            (F.col("brand") == ""),
            F.lit("Unknown")
        ).otherwise(F.col("brand"))
    )
    .withColumn(
        "status",
        F.when(
            F.col("status").isNull() |
            (F.col("status") == ""),
            F.lit("Unknown")
        ).otherwise(F.col("status"))
    )
    .withColumn(
        "unit_price",
        F.when(
            F.col("unit_price_raw").isNull() |
            (F.lower(F.col("unit_price_raw")) == "unknown") |
            (F.col("unit_price_raw") == ""),
            F.lit(None).cast("decimal(18,2)")
        ).otherwise(
            F.expr("""
                try_cast(
                    regexp_replace(
                        regexp_replace(
                            trim(unit_price_raw),
                            ',',
                            ''
                        ),
                        '[^0-9.-]',
                        ''
                    )
                    AS DECIMAL(18,2)
                )
            """)
        )
    )
    .withColumn(
        "created_date",
        F.coalesce(
            F.expr(
                "try_to_date(trim(created_date), 'yyyy-MM-dd')"
            ),
            F.expr(
                "try_to_date(trim(created_date), 'dd-MM-yyyy')"
            )
        )
    )
    .filter(
        F.col("product_id").isNotNull()
        & (F.col("product_id") != "")
    )
    .withColumn(
        "rn",
        F.row_number().over(
            Window
            .partitionBy("product_id")
            .orderBy(F.col("_ingested_at").desc())
        )
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
    .drop("unit_price_raw")
)

print("Rows:", product_base.count())

print(
    "Distinct product IDs:",
    product_base.select("product_id").distinct().count()
)

print(
    "NULL unit prices:",
    product_base
    .filter(F.col("unit_price").isNull())
    .count()
)

print(
    "NULL created dates:",
    product_base
    .filter(F.col("created_date").isNull())
    .count()
)

product_base.printSchema()

display(product_base.limit(10))

Rows: 800
Distinct product IDs: 800
NULL unit prices: 20
NULL created dates: 0
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- unit_price: decimal(18,2) (nullable = true)
 |-- status: string (nullable = true)
 |-- created_date: date (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)



product_id,product_name,category,brand,unit_price,status,created_date,_ingested_at
P00001,T-Shirt 1,Fashion,BrandC,null,discontinued,2025-02-15,2026-08-12T17:22:04.533Z
P00002,Bedsheet 2,Home,BrandC,null,active,2024-04-20,2026-08-12T17:22:04.533Z
P00003,Bedsheet 3,Home,BrandD,30753.26,active,2024-10-23,2026-08-12T17:22:04.533Z
P00004,Oil 4,Grocery,BrandB,68176.44,active,2024-03-07,2026-08-12T17:22:04.533Z
P00005,Oil 5,Unknown,BrandC,51796.39,active,2024-12-12,2026-08-12T17:22:04.533Z
P00006,Chair 6,Home,BrandC,10740.11,discontinued,2025-06-17,2026-08-12T17:22:04.533Z
P00007,Jeans 7,Fashion,BrandC,26020.02,active,2024-04-19,2026-08-12T17:22:04.533Z
P00008,Chair 8,Home,BrandB,8710.16,active,2023-03-23,2026-08-12T17:22:04.533Z
P00009,Cream 9,Beauty,BrandA,71925.36,discontinued,2023-02-22,2026-08-12T17:22:04.533Z
P00010,Chair 10,Home,BrandD,25422.83,discontinued,2025-12-02,2026-08-12T17:22:04.533Z


In [0]:
#  CREATE INITIAL PRODUCT SCD2

product_scd2 = (
    product_base
    .withColumn(
        "hash_value",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("product_id"), F.lit("")),
                F.coalesce(F.col("product_name"), F.lit("")),
                F.coalesce(F.col("category"), F.lit("")),
                F.coalesce(F.col("brand"), F.lit("")),
                F.coalesce(
                    F.col("unit_price").cast("string"),
                    F.lit("")
                ),
                F.coalesce(F.col("status"), F.lit("")),
                F.coalesce(
                    F.col("created_date").cast("string"),
                    F.lit("")
                )
            ),
            256
        )
    )
    .withColumn(
        "product_sk",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("product_id"),
                F.col("created_date").cast("string")
            ),
            256
        )
    )
    .withColumn(
        "effective_start_date",
        F.col("created_date")
    )
    .withColumn(
        "effective_end_date",
        F.to_date(F.lit("9999-12-31"))
    )
    .withColumn(
        "is_current",
        F.lit(True)
    )

    .select(
        "product_sk",
        "product_id",
        "product_name",
        "category",
        "brand",
        "unit_price",
        "status",
        "created_date",
        "effective_start_date",
        "effective_end_date",
        "is_current",
        "hash_value"
    )
)

print("Rows:", product_scd2.count())

print(
    "Distinct product IDs:",
    product_scd2
    .select("product_id")
    .distinct()
    .count()
)

print(
    "Current records:",
    product_scd2
    .filter(F.col("is_current") == True)
    .count()
)

product_scd2.printSchema()

display(product_scd2.limit(10))

Rows: 800
Distinct product IDs: 800
Current records: 800
root
 |-- product_sk: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- unit_price: decimal(18,2) (nullable = true)
 |-- status: string (nullable = true)
 |-- created_date: date (nullable = true)
 |-- effective_start_date: date (nullable = true)
 |-- effective_end_date: date (nullable = true)
 |-- is_current: boolean (nullable = false)
 |-- hash_value: string (nullable = true)



product_sk,product_id,product_name,category,brand,unit_price,status,created_date,effective_start_date,effective_end_date,is_current,hash_value
93eab7e6399c5797ccd798630d492193b3a011153ae82a4720332ec76e2870c4,P00001,T-Shirt 1,Fashion,BrandC,null,discontinued,2025-02-15,2025-02-15,9999-12-31,true,37120c249c59b6c941169b66abbb4d1e56c6102af81547ef9fd28f5f0d3f5a52
e52c6a242c13416dec94abf79e3dcd62fa25adbd83909bcb86775904fd8b2779,P00002,Bedsheet 2,Home,BrandC,null,active,2024-04-20,2024-04-20,9999-12-31,true,52f9dee10c4955d478ec07b94272ca2c8c66330df0c3930030331e5ed14bde6b
bc178bc540c9b4eb22bc3d1de0ea0e4c1f0657f0a1fc987bc293ac4040436601,P00003,Bedsheet 3,Home,BrandD,30753.26,active,2024-10-23,2024-10-23,9999-12-31,true,5db0e5cfdefde85e10ee8c1ef49f91444fd8bf641b469b29d0ea1219fbf02032
52762759b2401951921d0be2a7b4943c32070e3bd695600ac3561b46f475a4cf,P00004,Oil 4,Grocery,BrandB,68176.44,active,2024-03-07,2024-03-07,9999-12-31,true,17c127bb6978bef95801554a152120667bbed62180f4acda235c4032d9c0685d
73f5896b779d232f9c1de5c919ce52354cf527a3e541b4d443fc08b4f74a4d87,P00005,Oil 5,Unknown,BrandC,51796.39,active,2024-12-12,2024-12-12,9999-12-31,true,1732c3fed16cb7e756ad7c79cf6439ae8ed1f940fb8fe951e61959ff18c614d5
e9587bc2f66b8b76078503137b0ee89aa64ced65e8072654905ebed19e2a0ca5,P00006,Chair 6,Home,BrandC,10740.11,discontinued,2025-06-17,2025-06-17,9999-12-31,true,4d278499d173f62bbc44df94443ff2dbb29c9eda4735f9c4ec60a27f125c03ca
26f96323a877b90ef1571fbdab6456d32832173f839f683bc343e225567c8c1d,P00007,Jeans 7,Fashion,BrandC,26020.02,active,2024-04-19,2024-04-19,9999-12-31,true,c2dbeba344ab3a189662f695e17ac9797f3d5c8877c3c9f780a0dc7dc8ce9164
f235ff7d93007f992a870109e85326143ad14c106d44badd08322ba99ce54e2e,P00008,Chair 8,Home,BrandB,8710.16,active,2023-03-23,2023-03-23,9999-12-31,true,5f4908dd70c688eb9aff2567d1c7938b64cbd45e7248433be55e0c1d0bc79db6
338c8669a63f52d3edf3ca126e999ba14b0dd3def242d2aa50c8a06ad2aa9330,P00009,Cream 9,Beauty,BrandA,71925.36,discontinued,2023-02-22,2023-02-22,9999-12-31,true,ab6af1b7c4c3a4242aa67c6b1649f93484c901070701198cb18f60ba2d912084
334f73e275fe9c128de8c8472894d5a176d441e0aa7be445a93f1bfc52a94cf0,P00010,Chair 10,Home,BrandD,25422.83,discontinued,2025-12-02,2025-12-02,9999-12-31,true,c5fde7f3f9dd1aad15aa5987632d1102f56142aaadb3b96867cba9de047f5f4f


In [0]:
#  VALIDATE INITIAL PRODUCT SCD2

total_rows = product_scd2.count()

distinct_product_ids = (
    product_scd2
    .select("product_id")
    .distinct()
    .count()
)

current_rows = (
    product_scd2
    .filter(F.col("is_current") == True)
    .count()
)

historical_rows = (
    product_scd2
    .filter(F.col("is_current") == False)
    .count()
)

open_ended_rows = (
    product_scd2
    .filter(
        F.col("effective_end_date") ==
        F.to_date(F.lit("9999-12-31"))
    )
    .count()
)

null_surrogate_keys = (
    product_scd2
    .filter(F.col("product_sk").isNull())
    .count()
)

null_hash_values = (
    product_scd2
    .filter(F.col("hash_value").isNull())
    .count()
)

duplicate_surrogate_keys = (
    product_scd2
    .groupBy("product_sk")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Total rows:", total_rows)
print("Distinct product IDs:", distinct_product_ids)
print("Current rows:", current_rows)
print("Historical rows:", historical_rows)
print("Open-ended rows:", open_ended_rows)
print("Null surrogate keys:", null_surrogate_keys)
print("Null hash values:", null_hash_values)
print("Duplicate surrogate keys:", duplicate_surrogate_keys)

display(product_scd2.limit(10))

Total rows: 800
Distinct product IDs: 800
Current rows: 800
Historical rows: 0
Open-ended rows: 800
Null surrogate keys: 0
Null hash values: 0
Duplicate surrogate keys: 0


product_sk,product_id,product_name,category,brand,unit_price,status,created_date,effective_start_date,effective_end_date,is_current,hash_value
93eab7e6399c5797ccd798630d492193b3a011153ae82a4720332ec76e2870c4,P00001,T-Shirt 1,Fashion,BrandC,null,discontinued,2025-02-15,2025-02-15,9999-12-31,true,37120c249c59b6c941169b66abbb4d1e56c6102af81547ef9fd28f5f0d3f5a52
e52c6a242c13416dec94abf79e3dcd62fa25adbd83909bcb86775904fd8b2779,P00002,Bedsheet 2,Home,BrandC,null,active,2024-04-20,2024-04-20,9999-12-31,true,52f9dee10c4955d478ec07b94272ca2c8c66330df0c3930030331e5ed14bde6b
bc178bc540c9b4eb22bc3d1de0ea0e4c1f0657f0a1fc987bc293ac4040436601,P00003,Bedsheet 3,Home,BrandD,30753.26,active,2024-10-23,2024-10-23,9999-12-31,true,5db0e5cfdefde85e10ee8c1ef49f91444fd8bf641b469b29d0ea1219fbf02032
52762759b2401951921d0be2a7b4943c32070e3bd695600ac3561b46f475a4cf,P00004,Oil 4,Grocery,BrandB,68176.44,active,2024-03-07,2024-03-07,9999-12-31,true,17c127bb6978bef95801554a152120667bbed62180f4acda235c4032d9c0685d
73f5896b779d232f9c1de5c919ce52354cf527a3e541b4d443fc08b4f74a4d87,P00005,Oil 5,Unknown,BrandC,51796.39,active,2024-12-12,2024-12-12,9999-12-31,true,1732c3fed16cb7e756ad7c79cf6439ae8ed1f940fb8fe951e61959ff18c614d5
e9587bc2f66b8b76078503137b0ee89aa64ced65e8072654905ebed19e2a0ca5,P00006,Chair 6,Home,BrandC,10740.11,discontinued,2025-06-17,2025-06-17,9999-12-31,true,4d278499d173f62bbc44df94443ff2dbb29c9eda4735f9c4ec60a27f125c03ca
26f96323a877b90ef1571fbdab6456d32832173f839f683bc343e225567c8c1d,P00007,Jeans 7,Fashion,BrandC,26020.02,active,2024-04-19,2024-04-19,9999-12-31,true,c2dbeba344ab3a189662f695e17ac9797f3d5c8877c3c9f780a0dc7dc8ce9164
f235ff7d93007f992a870109e85326143ad14c106d44badd08322ba99ce54e2e,P00008,Chair 8,Home,BrandB,8710.16,active,2023-03-23,2023-03-23,9999-12-31,true,5f4908dd70c688eb9aff2567d1c7938b64cbd45e7248433be55e0c1d0bc79db6
338c8669a63f52d3edf3ca126e999ba14b0dd3def242d2aa50c8a06ad2aa9330,P00009,Cream 9,Beauty,BrandA,71925.36,discontinued,2023-02-22,2023-02-22,9999-12-31,true,ab6af1b7c4c3a4242aa67c6b1649f93484c901070701198cb18f60ba2d912084
334f73e275fe9c128de8c8472894d5a176d441e0aa7be445a93f1bfc52a94cf0,P00010,Chair 10,Home,BrandD,25422.83,discontinued,2025-12-02,2025-12-02,9999-12-31,true,c5fde7f3f9dd1aad15aa5987632d1102f56142aaadb3b96867cba9de047f5f4f


In [0]:
# — WRITE INITIAL PRODUCT SCD2

PRODUCT_SCD2_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.dim_product_scd2"
)

(
    product_scd2
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(PRODUCT_SCD2_TABLE)
)

print("Table:", PRODUCT_SCD2_TABLE)

display(
    spark.sql(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT product_id) AS distinct_products,
            SUM(
                CASE WHEN is_current = true THEN 1 ELSE 0 END
            ) AS current_rows
        FROM {PRODUCT_SCD2_TABLE}
    """)
)

Table: retail_demo.silver.dim_product_scd2


total_rows,distinct_products,current_rows
800,800,800


In [0]:
#  LOAD PRODUCT CDC

product_cdc = (
    spark.table(
        f"{CATALOG}.{RAW_SCHEMA}.bronze_products_incremental"
    )
    .select(
        "product_id",
        "product_name",
        "category",
        "brand",
        "unit_price",
        "status",
        "created_date",
        "effective_date",
        "operation",
        "ingest_ts"
    )

    # Clean strings
    .withColumn("product_id", F.trim(F.col("product_id")))
    .withColumn("product_name", F.trim(F.col("product_name")))
    .withColumn("category", F.trim(F.col("category")))
    .withColumn("brand", F.trim(F.col("brand")))
    .withColumn("unit_price_raw", F.trim(F.col("unit_price")))
    .withColumn("status", F.trim(F.col("status")))
    .withColumn("operation", F.trim(F.col("operation")))

    # Numeric price
    .withColumn(
        "unit_price",
        F.when(
            F.col("unit_price_raw").isNull() |
            (F.lower(F.col("unit_price_raw")) == "unknown") |
            (F.col("unit_price_raw") == ""),
            F.lit(None).cast("decimal(18,2)")
        ).otherwise(
            F.expr("""
                try_cast(
                    regexp_replace(
                        regexp_replace(
                            trim(unit_price_raw),
                            ',',
                            ''
                        ),
                        '[^0-9.-]',
                        ''
                    )
                    AS DECIMAL(18,2)
                )
            """)
        )
    )
    .drop("unit_price_raw")

    # Dates
    .withColumn(
        "created_date",
        F.coalesce(
            F.expr(
                "try_to_date(trim(created_date), 'yyyy-MM-dd')"
            ),
            F.expr(
                "try_to_date(trim(created_date), 'dd-MM-yyyy')"
            )
        )
    )
    .withColumn(
        "effective_date",
        F.coalesce(
            F.expr(
                "try_to_date(trim(effective_date), 'yyyy-MM-dd')"
            ),
            F.expr(
                "try_to_date(trim(effective_date), 'dd-MM-yyyy')"
            )
        )
    )

    .withColumn(
        "effective_start_date",
        F.coalesce(
            F.col("effective_date"),
            F.col("created_date")
        )
    )
)
print("CDC rows:", product_cdc.count())

print(
    "Distinct product IDs:",
    product_cdc
    .select("product_id")
    .distinct()
    .count()
)
print("\nOperation counts:")

display(
    product_cdc
    .groupBy("operation")
    .count()
    .orderBy("operation")
)
print("\nSample CDC records:")

display(product_cdc.limit(10))

CDC rows: 180
Distinct product IDs: 166

Operation counts:


operation,count
UPDATE,180



Sample CDC records:


product_id,product_name,category,brand,unit_price,status,created_date,effective_date,operation,ingest_ts,effective_start_date
P00173,Speaker 173,Electronics,BrandA,50560.38,discontinued,2025-09-11,2026-04-25,UPDATE,2026-08-11T17:55:36.528Z,2026-04-25
P00680,Perfume 680,Beauty,BrandC,40368.75,active,2024-12-24,2026-04-25,UPDATE,2026-08-11T17:55:36.528Z,2026-04-25
P00095,Smartphone 95,Electronics,BrandB,685.97,active,2023-10-15,2026-04-25,UPDATE,2026-08-11T17:55:36.528Z,2026-04-25
P00664,Cream 664,Beauty,BrandB,43522.35,discontinued,2023-03-22,2026-04-25,UPDATE,2026-08-11T17:55:36.528Z,2026-04-25
P00733,Jacket 733,Fashion,BrandA,26764.31,active,2023-07-31,2026-04-25,UPDATE,2026-08-11T17:55:36.528Z,2026-04-25
P00082,Jacket 82,Fashion,BrandA,52340.68,active,2025-01-26,2026-04-25,UPDATE,2026-08-11T17:55:36.528Z,2026-04-25
P00541,Laptop 541,Electronics,BrandA,77208.10,active,2025-12-22,2026-04-25,UPDATE,2026-08-11T17:55:36.528Z,2026-04-25
P00251,Jeans 251,Fashion,BrandA,88637.25,active,2025-11-28,2026-04-25,UPDATE,2026-08-11T17:55:36.528Z,2026-04-25
P00282,Chair 282,Home,BrandC,42367.76,active,2023-06-03,2026-04-25,UPDATE,2026-08-11T17:55:36.528Z,2026-04-25
P00142,Keyboard 142,Electronics,BrandC,16441.76,active,2025-06-06,2026-04-25,UPDATE,2026-08-11T17:55:36.528Z,2026-04-25


In [0]:
#  PRODUCT CDC HASH & CHANGE DETECTION

product_cdc_hashed = (
    product_cdc
    .withColumn(
        "hash_value",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("product_id"), F.lit("")),
                F.coalesce(F.col("product_name"), F.lit("")),
                F.coalesce(F.col("category"), F.lit("")),
                F.coalesce(F.col("brand"), F.lit("")),
                F.coalesce(
                    F.col("unit_price").cast("string"),
                    F.lit("")
                ),
                F.coalesce(F.col("status"), F.lit("")),
                F.coalesce(
                    F.col("created_date").cast("string"),
                    F.lit("")
                )
            ),
            256
        )
    )
)

current_products = (
    spark.table(
        f"{CATALOG}.{SILVER_SCHEMA}.dim_product_scd2"
    )
    .filter(F.col("is_current") == True)
    .select(
        "product_id",
        F.col("hash_value").alias("existing_hash"),
        F.col("product_sk").alias("existing_product_sk")
    )
)

product_cdc_changes = (
    product_cdc_hashed
    .join(
        current_products,
        on="product_id",
        how="left"
    )
    .withColumn(
        "change_type",
        F.when(
            F.col("existing_product_sk").isNull(),
            F.lit("INSERT")
        )
        .when(
            F.col("hash_value") != F.col("existing_hash"),
            F.lit("UPDATE")
        )
        .otherwise(
            F.lit("NO_CHANGE")
        )
    )
)
print(
    "Total CDC records:",
    product_cdc_changes.count()
)

display(
    product_cdc_changes
    .groupBy("change_type")
    .count()
    .orderBy("change_type")
)

Total CDC records: 180


change_type,count
UPDATE,180


In [0]:
#  FREEZE PRODUCT CDC CHANGE SET

product_changes_frozen = (
    product_cdc_changes
    .filter(
        F.col("change_type").isin(["INSERT", "UPDATE"])
    )
    .select(
        "product_id",
        "product_name",
        "category",
        "brand",
        "unit_price",
        "status",
        "created_date",
        "effective_date",
        "effective_start_date",
        "hash_value",
        "change_type",
        "ingest_ts"
    )
)

# Force evaluation before changing the target dimension
frozen_product_rows = product_changes_frozen.collect()
print(
    "Frozen change records:",
    len(frozen_product_rows)
)

product_changes_frozen = spark.createDataFrame(
    frozen_product_rows,
    schema=product_changes_frozen.schema
)

display(
    product_changes_frozen
    .groupBy("change_type")
    .count()
    .orderBy("change_type")
)

Frozen change records: 180


change_type,count
UPDATE,180


In [0]:
#  CHECK DUPLICATE PRODUCT CDC KEYS

duplicate_product_updates = (
    product_changes_frozen
    .groupBy("product_id")
    .agg(
        F.count("*").alias("update_records"),
        F.min("effective_start_date").alias("first_change_date"),
        F.max("effective_start_date").alias("last_change_date")
    )
    .filter(F.col("update_records") > 1)
    .orderBy(F.desc("update_records"))
)
print(
    "Products with multiple CDC records:",
    duplicate_product_updates.count()
)
print(
    "Total Product CDC records:",
    product_changes_frozen.count()
)
display(duplicate_product_updates)

Products with multiple CDC records: 13
Total Product CDC records: 180


product_id,update_records,first_change_date,last_change_date
P00639,3,2026-04-24,2026-04-26
P00541,2,2026-04-24,2026-04-25
P00456,2,2026-04-24,2026-04-25
P00187,2,2026-04-24,2026-04-25
P00297,2,2026-04-25,2026-04-26
P00675,2,2026-04-24,2026-04-25
P00625,2,2026-04-25,2026-04-26
P00509,2,2026-04-25,2026-04-26
P00583,2,2026-04-24,2026-04-25
P00326,2,2026-04-25,2026-04-26


In [0]:
# FINAL PRODUCT CDC SET

product_cdc_apply = (
    product_changes_frozen

    # Only actual changes
    .filter(
        F.col("change_type").isin(["INSERT", "UPDATE"])
    )
    .withColumn(
        "rn",
        F.row_number().over(
            Window
            .partitionBy(
                "product_id",
                "effective_start_date"
            )
            .orderBy(
                F.col("ingest_ts").desc()
            )
        )
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)
print(
    "Original Product changes:",
    product_changes_frozen.count()
)

print(
    "After same-day deduplication:",
    product_cdc_apply.count()
)

print("\nRecords by effective date:")

display(
    product_cdc_apply
    .groupBy("effective_start_date")
    .count()
    .orderBy("effective_start_date")
)

print("\nProducts with multiple change dates:")

display(
    product_cdc_apply
    .groupBy("product_id")
    .agg(
        F.countDistinct("effective_start_date")
        .alias("change_dates")
    )
    .filter(F.col("change_dates") > 1)
    .orderBy(F.desc("change_dates"))
)

Original Product changes: 180
After same-day deduplication: 180

Records by effective date:


effective_start_date,count
2026-04-24,60
2026-04-25,60
2026-04-26,60



Products with multiple change dates:


product_id,change_dates
P00639,3
P00005,2
P00072,2
P00187,2
P00297,2
P00326,2
P00456,2
P00483,2
P00509,2
P00541,2


In [0]:
# APPLY PRODUCT SCD2 CHRONOLOGICALLY


PRODUCT_SCD2_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.dim_product_scd2"
)

product_dim = DeltaTable.forName(
    spark,
    PRODUCT_SCD2_TABLE
)

product_cdc_dates = [
    row["effective_start_date"]
    for row in (
        product_cdc_apply
        .select("effective_start_date")
        .distinct()
        .orderBy("effective_start_date")
        .collect()
    )
]

for change_date in product_cdc_dates:
    print(f"\nProcessing: {change_date}")
    daily_product_cdc = (
        product_cdc_apply
        .filter(
            F.col("effective_start_date") == F.lit(change_date)
        )
        .select(
            "product_id",
            "product_name",
            "category",
            "brand",
            "unit_price",
            "status",
            "created_date",
            "effective_start_date",
            "hash_value"
        )
        .withColumn(
            "product_sk",
            F.sha2(
                F.concat_ws(
                    "||",
                    F.col("product_id"),
                    F.col("effective_start_date").cast("string"),
                    F.col("hash_value")
                ),
                256
            )
        )
        .withColumn(
            "effective_end_date",
            F.to_date(F.lit("9999-12-31"))
        )
        .withColumn(
            "is_current",
            F.lit(True)
        )
    )

    print(
        "Incoming:",
        daily_product_cdc.count()
    )

    (
        product_dim.alias("t")
        .merge(
            daily_product_cdc.alias("s"),
            """
            t.product_id = s.product_id
            AND t.is_current = true
            """
        )
        .whenMatchedUpdate(
            condition="t.hash_value <> s.hash_value",
            set={
                "effective_end_date":
                    "date_sub(s.effective_start_date, 1)",
                "is_current":
                    "false"
            }
        )
        .execute()
    )

    (
        product_dim.alias("t")
        .merge(
            daily_product_cdc.alias("s"),
            """
            t.product_id = s.product_id
            AND t.is_current = true
            AND t.hash_value = s.hash_value
            """
        )
        .whenNotMatchedInsert(
            values={
                "product_sk": "s.product_sk",
                "product_id": "s.product_id",
                "product_name": "s.product_name",
                "category": "s.category",
                "brand": "s.brand",
                "unit_price": "s.unit_price",
                "status": "s.status",
                "created_date": "s.created_date",
                "effective_start_date":
                    "s.effective_start_date",
                "effective_end_date":
                    "s.effective_end_date",
                "is_current":
                    "s.is_current",
                "hash_value":
                    "s.hash_value"
            }
        )
        .execute()
    )

    print("Completed:", change_date)

print("\nProduct SCD2 apply completed successfully.")


Processing: 2026-04-24
Incoming: 60
Completed: 2026-04-24

Processing: 2026-04-25
Incoming: 60
Completed: 2026-04-25

Processing: 2026-04-26
Incoming: 60
Completed: 2026-04-26

Product SCD2 apply completed successfully.


In [0]:
#FINAL PRODUCT SCD2 VALIDATION
product_dim_final = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.dim_product_scd2"
)
total_rows = product_dim_final.count()
distinct_products = (
    product_dim_final
    .select("product_id")
    .distinct()
    .count()
)
current_rows = (
    product_dim_final
    .filter(F.col("is_current") == True)
    .count()
)
historical_rows = (
    product_dim_final
    .filter(F.col("is_current") == False)
    .count()
)
open_ended_rows = (
    product_dim_final
    .filter(
        F.col("effective_end_date") ==
        F.to_date(F.lit("9999-12-31"))
    )
    .count()
)

null_surrogate_keys = (
    product_dim_final
    .filter(F.col("product_sk").isNull())
    .count()
)

duplicate_surrogate_keys = (
    product_dim_final
    .groupBy("product_sk")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

multiple_current_rows = (
    product_dim_final
    .groupBy("product_id")
    .agg(
        F.sum(
            F.when(F.col("is_current") == True, 1)
             .otherwise(0)
        ).alias("current_count")
    )
    .filter(F.col("current_count") != 1)
    .count()
)

print("Total dimension rows:", total_rows)
print("Distinct product IDs:", distinct_products)
print("Current rows:", current_rows)
print("Historical rows:", historical_rows)
print("Open-ended rows:", open_ended_rows)
print("Null surrogate keys:", null_surrogate_keys)
print("Duplicate surrogate keys:", duplicate_surrogate_keys)
print("Products with incorrect current-row count:", multiple_current_rows)

print("\nProducts with historical versions:")

display(
    product_dim_final
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
    .limit(20)
)

Total dimension rows: 980
Distinct product IDs: 800
Current rows: 800
Historical rows: 180
Open-ended rows: 800
Null surrogate keys: 0
Duplicate surrogate keys: 0
Products with incorrect current-row count: 0

Products with historical versions:


product_id,count
P00639,4
P00509,3
P00005,3
P00583,3
P00072,3
P00297,3
P00326,3
P00625,3
P00187,3
P00483,3
